# SIH26119 — simple Kaggle run

1. Import this small notebook (under 1 MB).
2. Upload **SIH26119_V2_source.zip** as a **private Kaggle dataset** and attach it
   to this notebook using **Add Input**. Any dataset name is fine.
3. Select **GPU** in Settings → Accelerator and enable **Internet**.
4. **Run All** (or Save Version → Save & Run All).
5. Download **sih26119_kaggle_results.zip** from the output and send it back.

The companion ZIP is required: it keeps the source and datasets out of the
notebook's size limit. Both zipped and unpacked input layouts are detected
automatically. No path edits or second virtual environment. The notebook uses
Kaggle's Python and builds a separate copy without changing the Windows project.

This is a quick iteration run: one timing repetition per configuration, not a
final speedup study. Without a usable GPU/compiler it runs CPU-only and labels
that clearly. If a build/test fails, send the diagnostic ZIP it produces.
CUDA covers convex box QPs only; LP/MILP/constrained QP run on CPU.

[Kaggle notebook settings help](https://www.kaggle.com/docs/notebooks)


In [ ]:
# 1. Load the companion source ZIP attached as a Kaggle input dataset.
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, traceback, zipfile
from IPython.display import display, FileLink

BASE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else BASE/"input"
WORK = Path(tempfile.mkdtemp(prefix="sih26119_", dir=BASE))
PYTHON = sys.executable
SNAPSHOT_SHA256 = "f840453501c39a975d78cec449adb120690a3a9d0ff7d8df6d6a010f1327bf8e"
MANIFEST_SHA256 = "eebce4dddab8317d349c0e553107ff613801cbd9d84f16835e7d2eed52c29ba9"
loaded = False

# Accept either the uploaded ZIP or files already unpacked by the dataset upload.
for source_zip in sorted(INPUT.rglob("SIH26119_V2_source.zip")):
    if hashlib.sha256(source_zip.read_bytes()).hexdigest() != SNAPSHOT_SHA256:
        continue
    with zipfile.ZipFile(source_zip) as archive:
        for entry in archive.infolist():
            if WORK.resolve() not in (WORK/entry.filename).resolve().parents:
                raise RuntimeError("Unsafe archive path")
        archive.extractall(WORK)
    loaded = True
    break
if not loaded:
    for marker in sorted(INPUT.rglob("SOURCE_MANIFEST.json")):
        if hashlib.sha256(marker.read_bytes()).hexdigest() != MANIFEST_SHA256:
            continue
        source_root = marker.parent.resolve()
        for name in json.loads(marker.read_text(encoding="utf-8")):
            source = (source_root/name).resolve()
            target = (WORK/name).resolve()
            if source_root not in source.parents or WORK.resolve() not in target.parents:
                raise RuntimeError("Unsafe dataset path")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        shutil.copy2(marker, WORK/"SOURCE_MANIFEST.json")
        loaded = True
        break
if not loaded:
    raise FileNotFoundError(
        "Attach the NEW SIH26119_V2_source.zip as a Kaggle input dataset, then Run All. "
        "The dataset name does not matter. An older ZIP will not match this notebook."
    )

manifest_path = WORK/"SOURCE_MANIFEST.json"
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == MANIFEST_SHA256
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected in manifest.items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest() == expected, "Source changed: "+name
(WORK/"SNAPSHOT.json").write_text(json.dumps({"snapshot_sha256": SNAPSHOT_SHA256}, indent=2))
os.chdir(WORK)
assert Path("scripts/test_result_parser.py").is_file()
print("Working directory:", Path.cwd())
print("Python:", PYTHON)
print("Source verified. Now run the next cell.")


## Run and get the ZIP

The following cell installs comparison packages in the current Python, builds
the solver, and runs the checks. Leave it running until it prints the ZIP path.
Known hard benchmark timeouts remain in the results; they are not hidden.


In [ ]:
# 2. Install, build, test, and produce the results ZIP. Uses Kaggle's Python.
PUBLIC_TIME_LIMIT = 30
SIZES = [1000, 100000, 1000000]
REPEATS = 1  # Quick feedback first; repeated performance measurements can follow.

RUN = Path(tempfile.mkdtemp(prefix="run_", dir=WORK))  # Safe to rerun this cell.
LOGS = RUN/"run_logs"
LOGS.mkdir()
RESULTS = RUN/"reports"/"kaggle_validation"
stage = "setup"
status = {"completed": False, "gpu_enabled": False, "python": PYTHON}

try:
    # Standard system checks. No nvidia-smi is a CPU-only run, not a crash.
    gpu_ok = False
    if shutil.which("nvidia-smi"):
        hardware = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
        (LOGS/"hardware.log").write_text(hardware.stdout+hardware.stderr, encoding="utf-8")
        gpu_ok = hardware.returncode == 0
    nvcc = shutil.which("nvcc")
    if not nvcc and Path("/usr/local/cuda/bin/nvcc").is_file():
        nvcc = "/usr/local/cuda/bin/nvcc"
    use_cuda = gpu_ok and bool(nvcc)
    status["gpu_enabled"] = use_cuda
    print("Mode:", "CPU + CUDA GPU" if use_cuda else "CPU ONLY (GPU or CUDA compiler unavailable)", flush=True)
    if not use_cuda:
        print("For GPU results: enable Settings > Accelerator > GPU, then Run All.", flush=True)

    configure = ["cmake", "-S", str(WORK), "-B", str(WORK/"build-v2"), "-DCMAKE_BUILD_TYPE=Release",
                 "-DSOVEREIGN_ENABLE_CUDA="+("ON" if use_cuda else "OFF")]
    if use_cuda:
        configure += ["-DCMAKE_CUDA_COMPILER="+nvcc]
        # Let CMake/compiler choose its default if capability querying is unavailable.
        capability = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                    capture_output=True, text=True, timeout=30)
        caps = capability.stdout.strip().splitlines()
        if capability.returncode == 0 and caps and caps[0].replace(".", "").isdigit():
            configure += ["-DCMAKE_CUDA_ARCHITECTURES="+caps[0].replace(".", "")]

    validation = [PYTHON, str(WORK/"scripts/validate_v2.py"), "--output", str(RESULTS),
                  "--limit", str(PUBLIC_TIME_LIMIT), "--sizes", *map(str, SIZES),
                  "--repeats", str(REPEATS), "--scale-limit", "120"]
    if use_cuda:
        validation.append("--cuda")
    commands = [
        ("install", [PYTHON, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                     "highspy==1.12.0", "osqp==1.0.5", "numpy", "scipy", "psutil", "certifi", "nvidia-ml-py"]),
        ("versions", [PYTHON, "-m", "pip", "freeze"]),
        ("compiler", ["g++", "--version"]),
        ("configure", configure),
        ("build", ["cmake", "--build", str(WORK/"build-v2"), "-j", "2"]),
        ("parser_tests", [PYTHON, str(WORK/"scripts/test_result_parser.py")]),
        ("validation", validation),
    ]
    if use_cuda:
        commands.insert(3, ("cuda_version", [nvcc, "--version"]))
    for stage, command in commands:
        print("Running:", stage, flush=True)
        log_path = LOGS/(stage+".log")
        with log_path.open("w", encoding="utf-8") as log:
            result = subprocess.run(command, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, timeout=3600)
        if stage != "versions":
            print(log_path.read_text(encoding="utf-8", errors="replace")[-6000:], flush=True)
        if result.returncode != 0:
            raise RuntimeError(f"{stage} exited with code {result.returncode}; see run_logs/{stage}.log")
    summary = json.loads((RESULTS/"summary.json").read_text())
    status["completed"] = True
    print(json.dumps(summary, indent=2))
    print("Run finished. Known benchmark failures, if any, are retained in the report.")
except Exception:
    status["failed_stage"] = stage
    status["error"] = traceback.format_exc()
    print(status["error"])
    print("Send the ZIP below; it contains the error details so we can fix the next iteration.")
finally:
    (LOGS/"RUN_STATUS.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    destination = BASE/"sih26119_kaggle_results.zip"
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder in (LOGS, RESULTS):
            if folder.exists():
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        archive.write(path, path.relative_to(RUN).as_posix())
        for name in ("SOURCE_MANIFEST.json", "SNAPSHOT.json"):
            if (WORK/name).is_file():
                archive.write(WORK/name, name)
    os.chdir(BASE)
    print("DOWNLOAD AND SEND BACK:", destination)
    display(FileLink(destination.name))
